# Ground-Truth Review & Export

Upload a **slim Excel** (`query`, `actual_output`/`expected_output`, `conversation_history`,
`extracted_metadata`), review each row as a card — query, answer, sources, summaries, and the
**circular images rendered inline** — tick the good ones, and click **Generate CSV**.

**Output CSV columns:** `query` (`{"query": ...}`), `expected_output` (`{"answer": ...}`),
`conversation_history` (JSON array or `[]`), `metadata` (the circulars JSON array).
UTF-8 with BOM so Arabic opens correctly in Excel.

**Images** load on demand per card (authenticated fetch → inline) so opening a large sheet
stays fast. Credentials are the same ones your explorer uses.

**How to use**
1. Cell 1 — setup + credentials.
2. Cell 2 — upload the slim `.xlsx` (or set `EXCEL_PATH` in cell 3).
3. Cell 3 — load.
4. Cell 4 — review cards: expand a card's images with **Show images**, tick the keepers.
5. Cell 5 — **Generate CSV**.

## 1. Setup & credentials

In [ ]:
# %pip install ipywidgets pandas openpyxl httpx --quiet
import pandas as pd, csv, json, base64, mimetypes, datetime as dt
from pathlib import Path
import httpx
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# --- credentials for authenticated image fetch (Approach B) ---
LANGFUSE_PUBLIC_KEY = "pk-lf-xxxxxxxx"
LANGFUSE_SECRET_KEY = "sk-lf-xxxxxxxx"
IMG_WIDTH = 900   # rendered image width (px) — big enough to read a circular page

OUT_DIR = Path("ground_truth_exports"); OUT_DIR.mkdir(exist_ok=True)

# one cached, authenticated client for images (verify=False, trust_env=False like the explorer)
_img_client = httpx.Client(auth=(LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY),
                           verify=False, trust_env=False,
                           timeout=httpx.Timeout(30.0, read=60.0), follow_redirects=True)
_img_cache = {}   # img_url -> data-URI (avoid re-downloading)

def fetch_image_datauri(url):
    """Authenticated fetch -> base64 data URI, cached. Returns None on failure."""
    if not url: return None
    if url in _img_cache: return _img_cache[url]
    try:
        r = _img_client.get(url); r.raise_for_status()
        ctype = (r.headers.get("content-type", "").split(";")[0]
                 or mimetypes.guess_type(url)[0] or "image/png")
        uri = f"data:{ctype};base64,{base64.b64encode(r.content).decode()}"
        _img_cache[url] = uri
        return uri
    except Exception as e:
        print(f"  ! image fetch failed ({type(e).__name__}) for {url[:80]}")
        return None

def _detect_columns(df):
    cols = {c.strip().lower(): c for c in df.columns}
    qcol = cols.get("query") or cols.get("input")
    acol = cols.get("actual_output") or cols.get("expected_output") or cols.get("answer")
    hcol = cols.get("conversation_history") or cols.get("history")
    mcol = cols.get("extracted_metadata") or cols.get("metadata")
    if qcol is None: qcol = list(df.columns)[0]
    return qcol, acol, hcol, mcol

def _wrap_query(text):
    if text is None: text = ""
    s = str(text).strip()
    if s.startswith("{"):
        try:
            o = json.loads(s)
            if isinstance(o, dict) and "query" in o:
                return json.dumps({"query": o["query"]}, ensure_ascii=False)
        except Exception: pass
    return json.dumps({"query": text}, ensure_ascii=False)

def _wrap_answer(text):
    if text is None: text = ""
    s = str(text).strip()
    if s.startswith("{"):
        try:
            o = json.loads(s)
            if isinstance(o, dict) and "answer" in o:
                return json.dumps({"answer": o["answer"]}, ensure_ascii=False)
        except Exception: pass
    return json.dumps({"answer": text}, ensure_ascii=False)

def _norm_history(val):
    if val is None: return "[]"
    s = str(val).strip()
    if s in ("", "[]"): return "[]"
    try:
        o = json.loads(s)
        if isinstance(o, list): return json.dumps(o, ensure_ascii=False) if o else "[]"
        return json.dumps(o, ensure_ascii=False)
    except Exception: return s

def _parse_circulars(val):
    """extracted_metadata cell -> list of circular dicts. Tolerates JSON array, or old
       pipe-delimited lines as a fallback."""
    if val is None: return []
    s = str(val).strip()
    if s in ("", "[]"): return []
    try:
        o = json.loads(s)
        if isinstance(o, list): return [c for c in o if isinstance(c, dict)]
    except Exception: pass
    # fallback: old "ts | filename | attachment_id | summary | img_url" lines
    out = []
    for line in s.split("\n"):
        parts = [p.strip() for p in line.split("|")]
        if len(parts) >= 1 and parts[0]:
            keys = ["timestamp", "filename", "attachment_id", "summary", "img_url"]
            out.append({keys[i]: parts[i] for i in range(min(len(parts), len(keys)))})
    return out

print("Setup ready. Exports ->", OUT_DIR.resolve())

## 2. Choose the Excel file
Use the uploader, **or** set `EXCEL_PATH` in cell 3.

In [ ]:
uploader = widgets.FileUpload(accept=".xlsx", multiple=False, description="Upload slim .xlsx")
display(uploader)
print("After picking a file, run cell 3. (Or set EXCEL_PATH in cell 3.)")

## 3. Load the rows

In [ ]:
EXCEL_PATH = ""   # optional: e.g. "langfuse_exports/xxx_slim.xlsx"

df = None
if EXCEL_PATH:
    df = pd.read_excel(EXCEL_PATH); print("Loaded:", EXCEL_PATH)
elif uploader.value:
    up = uploader.value[0] if isinstance(uploader.value, (list, tuple)) else list(uploader.value.values())[0]
    content = up["content"] if isinstance(up, dict) else up.content
    import io
    df = pd.read_excel(io.BytesIO(content))
    print("Loaded upload:", up.get("name") if isinstance(up, dict) else getattr(up, "name", "uploaded.xlsx"))
else:
    raise ValueError("No file. Use the uploader in cell 2, or set EXCEL_PATH here.")

df = df.fillna("")
QCOL, ACOL, HCOL, MCOL = _detect_columns(df)
print(f"Rows: {len(df)}")
print(f"Detected -> query: {QCOL!r} | answer: {ACOL!r} | history: {HCOL!r} | metadata: {MCOL!r}")
df.head(3)

## 4. Review cards

Each card shows the query, the answer (→ `expected_output`), conversation history (collapsed),
and each matched circular's details **with its image**. Click **Show images** on a card to
load its circular pictures (lazy, cached). Tick the rows that are good ground truth.

In [ ]:
row_checks = []

def _esc(t):
    return (str(t) if t is not None else "").replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")

def _rtl(text, size="14px"):
    return (f"<div dir='rtl' style='text-align:right;white-space:pre-wrap;font-size:{size};"
            f"font-family:\"Segoe UI\",Tahoma,Arial,sans-serif;line-height:1.7;"
            f"background:#fafafa;border-radius:6px;padding:8px;margin:4px 0'>{_esc(text)}</div>")

def _circular_text(c, idx):
    rows = []
    for label, key in [("التاريخ / timestamp","timestamp"),("الملف / filename","filename"),
                       ("attachment_id","attachment_id"),("الملخص / summary","summary")]:
        v = c.get(key)
        if v: rows.append(f"<tr><td style='color:#666;padding:2px 8px;white-space:nowrap'>{label}</td>"
                          f"<td dir='rtl' style='padding:2px 8px'>{_esc(v)}</td></tr>")
    return (f"<div style='font-weight:600;margin:6px 0 2px'>Circular {idx+1}</div>"
            f"<table style='border-collapse:collapse;font-size:13px;width:100%'>{''.join(rows)}</table>")

def build_review():
    global row_checks
    row_checks = []
    blocks = []
    for i in range(len(df)):
        q = df.iloc[i][QCOL]
        a = df.iloc[i][ACOL] if ACOL else ""
        hist = _norm_history(df.iloc[i][HCOL]) if HCOL else "[]"
        circs = _parse_circulars(df.iloc[i][MCOL]) if MCOL else []
        try: n_turns = len(json.loads(hist))
        except Exception: n_turns = 0

        chk = widgets.Checkbox(value=False, description="✓ Keep this row", indent=False,
                               layout=widgets.Layout(width="auto"))
        row_checks.append(chk)

        top = widgets.HTML(
            f"<div style='display:flex;gap:12px;flex-wrap:wrap'>"
            f"<div style='flex:1;min-width:280px'><b>Query</b>{_rtl(_query_text(q))}</div>"
            f"<div style='flex:1;min-width:280px'><b>Answer → expected_output</b>{_rtl(_answer_text(a))}</div>"
            f"</div>")

        # history accordion (collapsed)
        hist_html = widgets.HTML(_rtl(hist if n_turns else "[]", size="13px"))
        hist_acc = widgets.Accordion(children=[hist_html])
        hist_acc.set_title(0, f"Conversation history ({n_turns} turn(s))")
        hist_acc.selected_index = None  # collapsed

        # circulars: text now, images on demand
        circ_text = widgets.HTML("".join(_circular_text(c, j) for j in range(len(circs)) for c in [circs[j]])
                                 if circs else "<i style='color:#999'>No circulars</i>")
        img_out = widgets.Output()
        show_btn = widgets.Button(description=f"Show images ({len(circs)})", icon="image",
                                  disabled=(len(circs) == 0),
                                  layout=widgets.Layout(width="200px"))
        def _make_loader(circs_local, out_local, btn_local):
            def _load(_):
                btn_local.disabled = True; btn_local.description = "Loading…"
                with out_local:
                    clear_output()
                    for j, c in enumerate(circs_local):
                        url = c.get("img_url")
                        if not url:
                            display(HTML(f"<i style='color:#999'>Circular {j+1}: no img_url</i>")); continue
                        uri = fetch_image_datauri(url)
                        if uri:
                            display(HTML(f"<div style='margin:6px 0'><b>Circular {j+1}</b><br>"
                                         f"<img src='{uri}' style='max-width:{IMG_WIDTH}px;width:100%;"
                                         f"height:auto;border:1px solid #ccc;border-radius:6px'></div>"))
                        else:
                            display(HTML(f"<i style='color:#c00'>Circular {j+1}: image failed to load</i>"))
                btn_local.description = "Images loaded ✓"
            return _load
        show_btn.on_click(_make_loader(circs, img_out, show_btn))

        card = widgets.VBox(
            [widgets.HTML(f"<hr><div style='font-size:15px;font-weight:700'>Row {i}</div>"),
             chk, top, hist_acc, circ_text, show_btn, img_out],
            layout=widgets.Layout(border="2px solid #e0e0e0", border_radius="10px",
                                  padding="14px", margin="10px 0"))
        blocks.append(card)

    sel_all = widgets.Button(description="Select all", button_style="success")
    clr_all = widgets.Button(description="Clear all")
    counter = widgets.HTML()
    def _refresh(_=None):
        n = sum(c.value for c in row_checks)
        counter.value = f"<b style='font-size:15px'>{n}</b> / {len(row_checks)} selected"
    sel_all.on_click(lambda _: [setattr(c, "value", True) for c in row_checks])
    clr_all.on_click(lambda _: [setattr(c, "value", False) for c in row_checks])
    for c in row_checks: c.observe(_refresh, names="value")
    _refresh()
    bar = widgets.HBox([sel_all, clr_all, counter],
                       layout=widgets.Layout(border="1px solid #ccc", padding="8px",
                                            border_radius="8px", margin="0 0 8px 0"))
    display(bar); display(widgets.VBox(blocks))

# small helpers to show the *inner* text of wrapped query/answer cells in the UI
def _query_text(v):
    s = str(v).strip()
    if s.startswith("{"):
        try: return json.loads(s).get("query", s)
        except Exception: return s
    return s
def _answer_text(v):
    s = str(v).strip()
    if s.startswith("{"):
        try: return json.loads(s).get("answer", s)
        except Exception: return s
    return s

build_review()

## 5. Generate the ground-truth CSV

In [ ]:
gen_btn = widgets.Button(description="Generate CSV", button_style="primary",
                         icon="download", layout=widgets.Layout(width="220px", height="40px"))
gen_out = widgets.Output()

def _generate(_):
    with gen_out:
        clear_output()
        idx = [i for i, c in enumerate(row_checks) if c.value]
        if not idx:
            print("No rows selected. Tick at least one row in cell 4."); return
        src = df.iloc[idx].copy()
        records = []
        for _, r in src.iterrows():
            rec = {"query": _wrap_query(r[QCOL] if QCOL else "")}
            if ACOL: rec["expected_output"] = _wrap_answer(r[ACOL])
            rec["conversation_history"] = _norm_history(r[HCOL]) if HCOL else "[]"
            if MCOL:
                # keep metadata as the JSON array (re-serialize cleanly)
                rec["metadata"] = json.dumps(_parse_circulars(r[MCOL]), ensure_ascii=False)
            records.append(rec)
        col_order = [c for c in ["query", "expected_output", "conversation_history", "metadata"]
                     if c in records[0]]
        out = pd.DataFrame(records)[col_order]
        stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
        path = OUT_DIR / f"ground_truth_{stamp}.csv"
        out.to_csv(path, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_MINIMAL)
        print(f"Wrote {len(out)} row(s) -> {path}")
        print("Columns:", list(out.columns))
        display(out.head(min(10, len(out))))

gen_btn.on_click(_generate)
display(gen_btn, gen_out)

---
### Notes
- **Output CSV**: `query` (`{"query": ...}`), `expected_output` (`{"answer": ...}`),
  `conversation_history` (JSON array or `[]`), `metadata` (circulars JSON array with
  `timestamp, filename, attachment_id, summary, img_url`). UTF-8 **with BOM**.
- **Images** use authenticated fetch → inline base64 (Approach B), loaded **on demand** per
  card and **cached** by `img_url`, so a 100-row sheet won't download everything at once.
- `extracted_metadata` is read as a **JSON array**; old pipe-delimited sheets still parse via a
  fallback, so the notebook works with either format.
- Query/answer wrapping is **idempotent** — already-wrapped values pass through, no double-wrap.
- Re-running cell 4 resets selections and image state. Selections persist between cells 4 and 5.